# PyTorch Autograd: Automatic Differentiation from First Principles

## What is Autograd?

Every neural network trains by adjusting its weights to reduce a loss — and knowing *which direction* to adjust each weight requires the **gradient** of the loss with respect to that weight. Computing gradients by hand for a network with millions of parameters is not feasible.

**Autograd** (short for *automatic differentiation*) is PyTorch's engine for computing these gradients automatically. It works by recording every operation performed on tensors with `requires_grad=True` into a **dynamic computational graph**, then walking that graph backward (via `.backward()`) to compute gradients using the chain rule — this is exactly what powers **backpropagation**.

## What this notebook covers

1. Manual differentiation vs. PyTorch's `.backward()` on a simple function
2. Chain rule through composite functions (`sin(x**2)`)
3. A full worked example: gradients of Binary Cross-Entropy loss for logistic regression will be discussed
4. Gradients over vector inputs and gradient accumulation
5. Clearing gradients with `.zero_()`
6. Disabling gradient tracking: `requires_grad_(False)`, `.detach()`, and the errors you get when you try to backprop through a disconnected graph

**Suggested use:** run each cell in order and read the printed tensors carefully — notice `grad_fn=<...>` appearing on tensors once they're part of a tracked computation, and disappearing once tracking is turned off. That attribute is autograd's paper trail.

In [1]:
# y = x^2

def dy_dx(x):
  return 2*x

### Manual differentiation

For $y = x^2$, calculus gives $\frac{dy}{dx} = 2x$. At $x=3$, that's $6$ — matches the function above.

This works fine for a one-line function, but it doesn't scale: a real neural network's loss is a composition of thousands of operations across millions of parameters. Deriving and hand-coding each gradient is impractical. That's the problem autograd solves.

## 1. PyTorch Autograd Basics

To let PyTorch track a tensor's operations, create it with `requires_grad=True`. Every operation performed on it then gets recorded, building a computational graph you can walk backward with `.backward()`.

In [2]:
dy_dx(3)

6

In [3]:
import torch

In [4]:
x = torch.tensor(3.0, requires_grad=True)

In [5]:
y = x**2

In [6]:
x

tensor(3., requires_grad=True)

In [7]:
y

tensor(9., grad_fn=<PowBackward0>)

**What just happened:**
- `x = torch.tensor(3.0, requires_grad=True)` — tells autograd to track operations on `x`.
- `y = x**2` — PyTorch records this operation; note `y`'s `grad_fn=<PowBackward0>`, which is the "recipe" for computing $dy/dx$ during backprop.
- `y.backward()` — walks the graph backward and computes $dy/dx$ at $x=3$.
- `x.grad` — the computed gradient, stored on the leaf tensor `x`. It's `6.0`, matching our manual calculation from `dy_dx(3)`.

In [8]:
y.backward()

## 2. Chain Rule Through Composite Functions

Now a two-step composition: $y = x^2$, $z = \sin(y) = \sin(x^2)$.

By the chain rule: $\frac{dz}{dx} = \frac{dz}{dy}\cdot\frac{dy}{dx} = \cos(x^2)\cdot 2x$.

This is precisely what autograd does internally for *any* chain of operations — each `grad_fn` knows how to compute its local derivative, and `.backward()` multiplies them together along the graph.

In [9]:
x.grad

tensor(6.)

In [10]:
import math

def dz_dx(x):
    return 2 * x * math.cos(x**2)

In [11]:
dz_dx(4)

-7.661275842587077

In [12]:
x = torch.tensor(4.0, requires_grad=True)

In [13]:
y = x ** 2

In [14]:
z = torch.sin(y)

In [15]:
x

tensor(4., requires_grad=True)

In [16]:
y

tensor(16., grad_fn=<PowBackward0>)

In [17]:
z

tensor(-0.2879, grad_fn=<SinBackward0>)

In [18]:
z.backward()

In [19]:
x.grad

tensor(-7.6613)

In [20]:
y.grad

/tmp/ipykernel_2810/486760323.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  y.grad


**Note the `UserWarning` above.** PyTorch only retains `.grad` for **leaf tensors** (tensors you created directly, like `x`) — intermediate results like `y` are freed after `backward()` to save memory, since in a deep network you don't usually need gradients w.r.t. every intermediate activation. If you do need it, call `y.retain_grad()` *before* `backward()`.

## 3. A Real Example: Gradients for Logistic Regression

Let's compute the gradient of a **Binary Cross-Entropy (BCE) loss** with respect to a weight `w` and bias `b`, first entirely by hand using the chain rule, then again using autograd — to see that they agree.

Setup: a single input feature $x$, true label $y \in \{0, 1\}$, prediction $\hat{y} = \sigma(wx + b)$ (sigmoid), and loss $L = -\big(y\log\hat{y} + (1-y)\log(1-\hat{y})\big)$.

In [21]:
import torch

# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

**Manual chain rule breakdown:**

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z}\cdot\frac{\partial z}{\partial w}, \qquad \frac{\partial L}{\partial b} = \frac{\partial L}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z}\cdot\frac{\partial z}{\partial b}$$

where $z = wx+b$. Each factor above is a standard derivative (BCE w.r.t. prediction, sigmoid w.r.t. its input, and the linear layer w.r.t. its parameters). Multiplying them out gives the gradients printed below — this is backpropagation, done by hand, for one neuron.

In [22]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [23]:
# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [24]:
loss

tensor(6.7012)

In [25]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [26]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


In [27]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

In [28]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [29]:
w

tensor(1., requires_grad=True)

In [30]:
b

tensor(0., requires_grad=True)

In [31]:
z = w*x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [32]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [33]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [34]:
loss.backward()

In [35]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


**Same numbers, computed by autograd.** Compare `w.grad` and `b.grad` above to the manually derived `dL_dw` and `dL_db` a few cells up — they match. This is the core promise of autograd: you write the forward pass, and it derives the backward pass for you, correctly, no matter how deep or complex the computation graph gets.

## 4. Gradients Over Vectors

Autograd isn't limited to scalars — it works the same way when `x` is a vector (or any tensor). The one requirement: `.backward()` can only be called directly on a **scalar** output (like a loss), which is why we reduce with `.mean()` below before calling it.

In [36]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

In [37]:
x

tensor([1., 2., 3.], requires_grad=True)

In [38]:
y = (x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [39]:
y.backward()

In [40]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

For $y = \text{mean}(x^2) = \frac{1}{3}\sum x_i^2$, $\frac{\partial y}{\partial x_i} = \frac{2x_i}{3}$ — which is exactly the elementwise result in `x.grad` above (`[0.667, 1.333, 2.000]` for `x = [1, 2, 3]`).

## 5. Clearing Gradients

**Important:** PyTorch **accumulates** gradients into `.grad` by default — every call to `.backward()` *adds* to whatever's already there rather than replacing it. This is intentional (it's what lets you accumulate gradients across multiple mini-batches), but it means that in a training loop you must explicitly zero out gradients before each new `.backward()` call, usually via `optimizer.zero_grad()` — otherwise gradients from previous steps silently leak into the current one.

In [41]:
# clearing grad
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [42]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [43]:
y.backward()

In [44]:
x.grad

tensor(4.)

In [45]:
x.grad.zero_()

tensor(0.)

`x.grad.zero_()` resets the accumulated gradient back to `0`, confirmed by the output above.

## 6. Disabling Gradient Tracking

Not every tensor needs its graph tracked — e.g. once training is done and you're just running inference, tracking wastes memory and compute. PyTorch gives you three ways to disable it:

1. **`x.requires_grad_(False)`** — turns off tracking on `x` in-place, permanently (until turned back on).
2. **`x.detach()`** — returns a *new* tensor sharing the same data but detached from the computation graph; `x` itself is untouched.
3. **`torch.no_grad()`** — a context manager that disables tracking for everything inside its block, without modifying any tensor's `requires_grad` flag.

Let's see what happens when you try to backpropagate through a tensor that isn't tracked.

In [46]:
# disable gradient tracking
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [47]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [48]:
y.backward()

In [49]:
x.grad

tensor(4.)

In [50]:
# option 1 - requires_grad_(False)
# option 2 - detach()
# option 3 - torch.no_grad()

In [51]:
x.requires_grad_(False)

tensor(2.)

In [52]:
x

tensor(2.)

In [53]:
y = x ** 2

In [54]:
y

tensor(4.)

**Why this fails:** once `x.requires_grad_(False)` is called, subsequent operations like `y = x ** 2` are no longer recorded — `y` has no `grad_fn`, so there's no graph to walk backward through. Hence `RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn`.

`detach()` is the non-destructive alternative: `x` still has `requires_grad=True` and can still be backpropagated through (see `y = x**2` below), while `z` is a separate, gradient-free snapshot of the same value — useful when you want to use a tensor's value without it dragging the rest of the graph along.

In [55]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [56]:
z = x.detach()
z

tensor(2.)

In [57]:
y = x ** 2

In [58]:
y

tensor(4., grad_fn=<PowBackward0>)

In [59]:
y1 = z ** 2
y1

tensor(4.)

In [60]:
y.backward()

**Same failure mode as before**, just via a different path: `y1 = z ** 2` was built from the detached `z`, so `y1` has no `grad_fn` either. Meanwhile `y = x ** 2` (built from the original, still-tracked `x`) backpropagated successfully in the cell above. This contrast is the whole point of `detach()` — it lets you branch off a value without breaking gradient flow for the rest of your computation.

In [61]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [62]:
y = x ** 2

In [63]:
y

tensor(4., grad_fn=<PowBackward0>)

In [64]:
y.backward()

## Conclusion

This notebook walked through the core mental model of PyTorch autograd:

- **Tracking**: tensors created with `requires_grad=True` (or derived from them) get every operation recorded into a computational graph, visible via the `grad_fn` attribute.
- **Backward pass**: calling `.backward()` on a scalar walks that graph in reverse, applying the chain rule at each step to compute $\partial(\text{output})/\partial(\text{leaf tensor})$.
- **Gradient storage**: results land in `.grad`, but only for **leaf** tensors, and they **accumulate** across calls — remember to `.zero_()` between training steps.
- **Turning it off**: `requires_grad_(False)`, `.detach()`, and `torch.no_grad()` all stop tracking, each suited to a different situation (permanent vs. one-off snapshot vs. scoped block).

The logistic-regression example is the key takeaway: hand-deriving $\partial L/\partial w$ via the chain rule and getting the *exact same number* as `w.grad` is proof that autograd isn't magic — it's mechanically applying the same calculus you'd do by hand, just automatically and for graphs far too large to differentiate manually. That's what makes training deep networks with millions of parameters tractable.

**Try next:** extend the logistic regression example to a small batch of inputs instead of one scalar `x`, or wrap the forward/backward/`zero_grad` steps into an explicit training loop with `torch.optim.SGD`.